In [167]:
import pandas as pd
import numpy as np
import datetime
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
import requests
import re

## 1. We imported the datasets in this section

In [168]:
#Rainfall dataset
data1 = pd.read_csv("rain.csv")

In [169]:
data1.head()

,State,City,Latitude,Longitude,Onset date,Season end,Date,Season Length Days,Annual Rainfall mm
0,Abia,Aba,5.10,7.35,8-Mar-2017,12-Dec-2017,2017,276,2505
1,Abia,Abiriba,5.70,7.73,15-Mar-2017,7-Dec-2017,2017,263,2220
2,Abia,Arochukwu,5.38,7.92,11-Mar-2017,10-Dec-2017,2017,270,2369
3,Abia,Arochukwu,5.53,7.76,13-Mar-2019,12-Dec-2019,2019,275,2201
4,Abia,Bende,5.55,7.63,14-Mar-2017,8-Dec-2017,2017,266,2289


In [170]:
#Food price dataset
data2 = pd.read_csv("all.csv")

In [171]:
#inflation rate dataset
data3 = pd.read_csv("InflationRates2019.csv")

## 2. Here we try to fish issues and clean them up

Since we will need to merged the dataset, we have to make sure there are some allignment between them. 

>1. First we have make sure the years tally. To do this, we removed 2021 and 2022 from the food price dataset (data2)

>2. We have to make sure the states in each datasets tally. To do this, we use the assert method to check if the list of states in each data set (data1 and data2) are thesame

>3. We have to ...

In [172]:
#removing 2021 and 2022
data2 = data2[~data2['Date'].str.startswith(('2021', '2022'))]

In [173]:
#assert list(data1["State"].unique()) == list(data2["states"].unique())

In [174]:
data1.State.unique() 

array(['Abia', 'Adamawa', 'Akwa Ibom', 'Anambra', 'Bauchi', 'Bayelsa',
       'Benue', 'Borno', 'Cross River', 'Delta', 'Ebonyi', 'Edo', 'Ekiti',
       'Enugu', 'Gombe', 'Imo', 'Jigawa', 'Kaduna', 'Kano', 'Katsina',
       'Kebbi', 'Kogi', 'Kwara', 'Lagos', 'Nasarawa', 'Niger', 'Ogun',
       'Ondo', 'Osun', 'Oyo', 'Plateau', 'Rivers', 'Sokoto', 'Taraba',
       'Yobe', 'Zamfara', 'FCT', 'River'], dtype=object)

In [175]:
data2.states.unique()

array(['Benue', 'Borno', 'Cross River', 'Delta', 'Ebonyi', 'Edo', 'Ekiti',
       'Enugu', 'Gombe', 'Imo', 'Jigawa', 'Kaduna', 'Kano', 'Katsina',
       'Kebbi', 'Kogi', 'Kwara', 'Lagos', 'Nassarawa', 'Niger', 'Ogun',
       'Ondo', 'Osun', 'Oyo', 'Plateau', 'Rivers', 'Sokoto', 'Taraba',
       'Yobe', 'Zamfara', 'Bayelsa', 'Bauchi', 'Anambra', 'Akwa Ibom',
       'Adamawa', 'Abia', 'Abuja'], dtype=object)

We notices some discrepancy in the spelling of 3 states (`Nasarawa`, `FCT` and `River`)

In [176]:
#data1["State"] = data1["State"].str.replace("FCT", "Abuja")
data1.loc[data1['State']== "FCT", "State"] = "Abuja"
data1.loc[data1['State']== "River", "State"] = "Rivers"
data1.loc[data1['State']== "Nasarawa", "State"] = "Nassarawa"

We corrected the errors/discrepancy, sorted the state columns for the two datasets and assert their similiarity.

In [177]:
data1 = data1.sort_values(by='State')

In [178]:
data2 = data2.sort_values(by='states')


In [179]:
assert list(data1["State"].unique()) == list(data2["states"].unique())

### Now lets scrape some data from the internet

In [180]:
urls = ["https://geokeo.com/database/state/ng/1/", "https://geokeo.com/database/state/ng/2/"]
dfs = pd.DataFrame()

for url in urls:
    pages = requests.get(url)
    soups = BeautifulSoup(pages.text, 'html')

    tables  = soups.find_all('table', class_ = "table table-hover table-bordered")[0]
    heading = tables.find('tr')
    title = [x.text.strip() for x in heading]
    result_list = [element for element in title if element != '']

    
    col_datas = tables.find_all('tr')
    data_rows = []
    for col in col_datas[1:]:
        row_datas = col.find_all('td')
        entrys = [data.text for data in row_datas]
        data_rows.append(entrys)

    page_df = pd.DataFrame(data_rows, columns=result_list)
    dfs = pd.concat([dfs, page_df], ignore_index=True)
    


In [181]:
# Some cleanings
    
#result_df['Name'] = result_df['States'].str.replace(r'\(\d+\)', '')
#df['States'] = df['States'].str.strip()

dfs.loc[dfs['Name']== "Federal Capital Territory", "Name"] = "Abuja"

dfs= dfs.rename(columns = {"Name":"State"})

df1 = dfs.drop(columns = ["#","Country", "Other Language Names"])

In [182]:
df1.head()
df1.to_csv("./long_lat_state.csv", index=False)

In [183]:
df = pd.read_csv("long_lat_state.csv")

In [184]:
df.head()

,State,Latitude,Longitude
0,Abia,5.454095,7.515307
1,Adamawa,9.512977,12.388189
2,Akwa Ibom,4.940864,7.841227
3,Anambra,6.218314,6.953184
4,Bauchi,10.622828,10.028775


## Next we want to limit the rainfall dataset (data1) such that only the states are used and not the local goverment.

In [185]:
food_df = data1.copy()

In [186]:
food_df = food_df[["State", "Date", "Season Length Days", "Annual Rainfall mm"]]

In [187]:
mean_df = food_df.pivot_table(index=['State', 'Date'], 
                             values=['Season Length Days', 'Annual Rainfall mm'], aggfunc='mean')

In [188]:
mean_df =mean_df.reset_index()

In [189]:
mean_df["Date"].unique()

array([2017, 2019, 2020])

In [190]:
mean_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 111 entries, 0 to 110
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   State               111 non-null    object 
 1   Date                111 non-null    int64  
 2   Annual Rainfall mm  111 non-null    float64
 3   Season Length Days  111 non-null    float64
dtypes: float64(2), int64(1), object(1)
memory usage: 3.6+ KB


We observed that 2018 is missing in this data. luckily we were able to download it from the site [Statista](https://www.statista.com/statistics/1264326/annual-rainfall-in-nigeria-by-state/)

In [191]:
seasonal_2018_interpolation = food_df.pivot_table(index=['State'], 
                             values=['Season Length Days'], aggfunc='mean')
inte_df = seasonal_2018_interpolation.reset_index()


# We need just the seasonal length days 

inte_list = inte_df[["Season Length Days"]]


In [192]:
# read in the dataset for 2018 rainfall
df_2018 = pd.read_excel("annual_rainfall_2018.xlsx", sheet_name = "data2")

df_2018["Season Length Days"] = inte_list

In [193]:
#df_2018["Date"] = pd.to_datetime(df_2018['Date'], format='%Y')
#df_2018["Date"] = df_2018["Date"].dt.year

df_2018.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36 entries, 0 to 35
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Date                36 non-null     int64  
 1   State               36 non-null     object 
 2   Annual Rainfall mm  36 non-null     int64  
 3   Season Length Days  36 non-null     float64
dtypes: float64(1), int64(2), object(1)
memory usage: 1.2+ KB


In [194]:
new_df = mean_df.merge(df_2018, on=['Date', 'State', 'Annual Rainfall mm', 'Season Length Days'], how='outer', )

/var/folders/_9/z4llrcqd0y122r7_nwdcm5900000gn/T/ipykernel_55253/1337846014.py:1: UserWarning:

You are merging on int and float columns where the float values are not equal to their int representation.



In [195]:
new_df.isna().sum()

State                 0
Date                  0
Annual Rainfall mm    0
Season Length Days    0
dtype: int64

In [196]:
new_df.State.unique()

array(['Abia', 'Abuja', 'Adamawa', 'Akwa Ibom', 'Anambra', 'Bauchi',
       'Bayelsa', 'Benue', 'Borno', 'Cross River', 'Delta', 'Ebonyi',
       'Edo', 'Ekiti', 'Enugu', 'Gombe', 'Imo', 'Jigawa', 'Kaduna',
       'Kano', 'Katsina', 'Kebbi', 'Kogi', 'Kwara', 'Lagos', 'Nassarawa',
       'Niger', 'Ogun', 'Ondo', 'Osun', 'Oyo', 'Plateau', 'Rivers',
       'Sokoto', 'Taraba', 'Yobe', 'Zamfara', 'FCT', 'Nasarawa'],
      dtype=object)

In [197]:
df

,State,Latitude,Longitude
0,Abia,5.454095,7.515307
1,Adamawa,9.512977,12.388189
2,Akwa Ibom,4.940864,7.841227
3,Anambra,6.218314,6.953184
4,Bauchi,10.622828,10.028775
5,Bayelsa,4.762979,6.028898
6,Benue,7.350575,8.777288
7,Borno,12.187539,13.308003
8,Cross River,5.867197,8.520477
9,Delta,5.527306,6.178417


In [198]:
final_df = new_df.merge(df, on=['State'])

In [199]:
final_df

,State,Date,Annual Rainfall mm,Season Length Days,Latitude,Longitude
0,Abia,2017,2344.193548,269.193548,5.454095,7.515307
1,Abia,2019,2176.058824,273.529412,5.454095,7.515307
2,Abia,2020,2127.294118,276.764706,5.454095,7.515307
3,Abia,2018,2439.000000,272.307692,5.454095,7.515307
4,Abuja,2017,1136.117647,200.000000,8.831123,7.172467
...,...,...,...,...,...,...
138,Zamfara,2017,791.533333,131.733333,12.007900,6.419143
139,Zamfara,2019,434.714286,125.714286,12.007900,6.419143
140,Zamfara,2020,707.285714,131.142857,12.007900,6.419143
141,Zamfara,2018,891.000000,121.955556,12.007900,6.419143


In [200]:
data3["year"] =data3.index

In [201]:
data3 = data3.rename(columns = {"Year":"Month", "Month":"All Items (Year on Change)",
                                "All Items (Year on Change)": "All Items (12 Months Avg. Change)",
                               "All Items (12 Months Avg. Change)": "Food (Year on Change)/1",
                                "Food (Year on Change)/1":"Food (12 Months Avg. Change)/1", 
                                "Food (12 Months Avg. Change)/1": "All Items Less Farm Produce (Year on Change)/2",
                               "All Items Less Farm Produce (Year on Change)/2":"All Items Less Farm Produce (12 Months Avg. Change)/2",
                               "All Items Less Farm Produce (12 Months Avg. Change)/2":"All Items Less Farm Produce and Energy (Year on Change)/3"})

In [202]:
#selecting only required columns
data4 = data3[["Month","All Items (Year on Change)", "All Items (12 Months Avg. Change)",
              "Food (12 Months Avg. Change)/1", "Food (12 Months Avg. Change)/1"]]
data5 = data4.reset_index()
data5 = data5.rename(columns = {"index":"Year"})

In [203]:
#Extracting 2017 to 2020 only
data6 = data5.loc[(data5["Year"]>=2017) & (data5["Year"]<=2020)]
data6 = data6.reset_index()
data6 =data6.drop(columns = ["index"])

In [204]:
#splitting the date column into year and month in the food price dataset
data2['Year'] = data2['Date'].str.extract(r'(\d{4})')  
data2['Month'] = data2['Date'].str.extract(r'M(\d+)')

In [205]:
#Dropping irrelevant columns from the food price dataset
data7= data2.copy()
data7 = data7.drop(columns = ["Date", "Unit"])

In [206]:
#converting year and month into integer in the foodprice dataset
data7["Year"] = data7["Year"].astype(int)
data7["Month"] = data7["Month"].astype(int)

In [207]:
data7

,states,crops,Value,Year,Month
29988,Abia,Onion bulb,231.095570,2018,12
30122,Abia,Maize grain yellow sold loose,258.950617,2019,4
30123,Abia,Maize grain yellow sold loose,244.631154,2019,5
30124,Abia,Maize grain yellow sold loose,168.518519,2019,6
30125,Abia,Maize grain yellow sold loose,137.847222,2019,7
...,...,...,...,...,...
24783,Zamfara,Tomato,320.418572,2018,7
24784,Zamfara,Tomato,341.850253,2018,8
24785,Zamfara,Tomato,357.238564,2018,9
24778,Zamfara,Tomato,248.330897,2018,2


In [208]:
#First merging:merging the food price dataset with the inflation data
merged_df = data7.merge(data5, on=['Year', 'Month'])

In [209]:
#Sort the merged data by state Alphabetically  
merged_df = merged_df.sort_values(by='states')

In [210]:
#Quick renaming of the state and the date columns of rainfall data
data = final_df.rename(columns= {"State": "states", "Date":"Year"})
data.head()

,states,Year,Annual Rainfall mm,Season Length Days,Latitude,Longitude
0,Abia,2017,2344.193548,269.193548,5.454095,7.515307
1,Abia,2019,2176.058824,273.529412,5.454095,7.515307
2,Abia,2020,2127.294118,276.764706,5.454095,7.515307
3,Abia,2018,2439.000000,272.307692,5.454095,7.515307
4,Abuja,2017,1136.117647,200.000000,8.831123,7.172467


In [211]:
#Second merging: merging the first merged dataset with rainfall dataset
merged_df2 = data.merge(merged_df, on=['states', 'Year'])

merged_df2.shape

(22152, 13)

In [212]:
#Tidying up and renaming the columns
merged_df3 = merged_df2.rename(columns =
                               {"Value":"Price/Kg (Naira)", "All Items (Year on Change)":"inflation_rate(all_item annual)",
                             "All Items (12 Months Avg. Change)":"inflation_rate(all_item monthly avg.)",
                             "Food (Year on Change)/1":"inflation_rate(food_item annual)",
                                "Food (12 Months Avg. Change)/1": "inflation_rate(food_item monthly avg.)"}) 
                                

In [234]:
merged_df3.to_csv("./food_price_dataset")

In [235]:
price_df = pd.read_csv("food_price_dataset")

# Lets explore the new datasets

In [236]:
# Exploring the first few columns 
price_df.head()

,Unnamed: 0,states,Year,Annual Rainfall mm,Season Length Days,Latitude,Longitude,crops,Price/Kg (Naira),Month,inflation_rate(all_item annual),inflation_rate(all_item monthly avg.),inflation_rate(food_item monthly avg.),inflation_rate(food_item monthly avg.).1
0,0,Abia,2017,2344.193548,269.193548,5.454095,7.515307,"Beans brown,sold loose",458.333333,7,16.05,17.47,18.25,18.25
1,1,Abia,2017,2344.193548,269.193548,5.454095,7.515307,Rice Medium Grained,439.665224,7,16.05,17.47,18.25,18.25
2,2,Abia,2017,2344.193548,269.193548,5.454095,7.515307,"Gari white,sold loose",394.147465,7,16.05,17.47,18.25,18.25
3,3,Abia,2017,2344.193548,269.193548,5.454095,7.515307,Beans:white black eye. sold loose,463.793103,7,16.05,17.47,18.25,18.25
4,4,Abia,2017,2344.193548,269.193548,5.454095,7.515307,"Gari yellow,sold loose",428.156682,7,16.05,17.47,18.25,18.25


In [221]:
# Checking out the columns and enteries as well as data types
price_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 22152 entries, 0 to 22151
Data columns (total 13 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   states                                  22152 non-null  object 
 1   Year                                    22152 non-null  int64  
 2   Annual Rainfall mm                      22152 non-null  float64
 3   Season Length Days                      22152 non-null  float64
 4   Latitude                                22152 non-null  float64
 5   Longitude                               22152 non-null  float64
 6   crops                                   22152 non-null  object 
 7   Price/Kg (Naira)                        22152 non-null  float64
 8   Month                                   22152 non-null  int64  
 9   inflation_rate(all_item annual)         22152 non-null  float64
 10  inflation_rate(all_item monthly avg.)   22152 non-null  fl

In [222]:
#Exploring some statistical summaries of the columns
price_df.describe()

,Year,Annual Rainfall mm,Season Length Days,Latitude,Longitude,Price/Kg (Naira),Month,inflation_rate(all_item annual),inflation_rate(all_item monthly avg.),inflation_rate(food_item monthly avg.),inflation_rate(food_item monthly avg.)
count,22152.000000,22152.000000,22152.000000,22152.000000,22152.000000,22152.000000,22152.00000,22152.000000,22152.000000,22152.000000,22152.000000
mean,2018.507042,1377.226042,207.926831,8.479617,7.371458,265.193374,6.50000,13.340129,13.699695,15.840728,15.840728
std,1.124318,621.059814,53.868263,2.538329,2.491175,103.378472,3.45213,2.215530,2.367263,2.128368,2.128368
min,2017.000000,314.000000,109.478261,4.762979,3.438929,70.356125,1.00000,11.020000,11.270000,13.340000,13.340000
25%,2017.000000,801.250000,161.636364,6.218314,5.315220,195.566308,3.75000,11.370000,11.540000,13.860000,13.860000
50%,2019.000000,1312.452381,223.343750,7.904547,7.119716,249.079378,6.50000,12.400000,12.660000,15.360000,15.360000
75%,2020.000000,1851.190476,253.700000,10.622828,8.777288,322.580645,9.25000,15.750000,16.440000,17.750000,17.750000
max,2020.000000,2695.100000,288.130435,13.061119,13.308003,985.075365,12.00000,18.720000,17.630000,19.620000,19.620000


In [223]:
#Checking out and dropping duplicated values 
price_df.duplicated().sum()
#price_df = price_df.drop_duplicates()

0

In [225]:
#There are no missing values 
price_df.isnull().sum()

states                                    0
Year                                      0
Annual Rainfall mm                        0
Season Length Days                        0
Latitude                                  0
Longitude                                 0
crops                                     0
Price/Kg (Naira)                          0
Month                                     0
inflation_rate(all_item annual)           0
inflation_rate(all_item monthly avg.)     0
inflation_rate(food_item monthly avg.)    0
inflation_rate(food_item monthly avg.)    0
dtype: int64

In [226]:
# Creating a new column called month name
def map_month(month_num):
    month_names = {1: 'Jan', 2: 'Feb', 3: 'March', 4: 'Apr', 5: 'May', 6: 'Jun', 7: 'Jul',
                  8: 'Aug', 9: 'Sept', 10: 'Oct', 11: 'Nov', 12: 'Dec'}
    return month_names.get(month_num, 'Invalid Month')

# Create a new column with month names
price_df['Month Name'] = price_df['Month'].apply(lambda x: map_month(x))

In [227]:
price_df['Year'] = price_df.Year.astype(str)

In [228]:
#Checking the final shape of the dataset
price_df.shape

(22152, 14)

# Let's commence visualization 

In [229]:
# Checking the number of unique values for the crops

uniques = price_df.crops.unique()
for i, x in enumerate(uniques):
    print(f"{i} -------  {x}")

0 -------  Beans brown,sold loose
1 -------  Rice Medium Grained
2 -------  Gari white,sold loose
3 -------  Beans:white black eye. sold loose
4 -------  Gari yellow,sold loose
5 -------  Onion bulb
6 -------  Broken Rice (Ofada)
7 -------  Tomato
8 -------  Plantain(unripe)
9 -------  Maize grain white sold loose
10 -------  Yam tuber
11 -------  Maize grain yellow sold loose
12 -------  Plantain(ripe)


In [233]:
# Lets check the rainfall distribution 


fig = px.histogram(price_df["Annual Rainfall mm"],
                   nbins=40, title='Annual rainfall distribution', 
                   labels={'x': 'Rainfall', 'y': 'Frequency'})
fig.show()

In [ ]:
# Let us see the annual rainfall for each year

#fig = px.bar(price_df, x='Year', y='Annual Rainfall mm', title='Year vs Annual Rainfall')

# Show the plot
#fig.show()